# Step 6 — Step 2: export locked CodeT5 caches for the anchor runner (WSL)

This notebook reads the committed `step6_lock.json`, rechecks the frozen raw identities and active prompt retrieval provenance, then writes ID-aligned NPZ adapters (`ids`, `X`) from the existing `vectors.npy` files. It does not run CodeT5, rebuild embeddings, retrieve demos, or fit models.

Lock commit: `e8f3cdc` (`docs:lock-step6-anchor-inputs`). Expected lock SHA-256 is pinned below.

The NPZ files are inputs for SUP-LR and SUP-MLP. SUP-METRIC uses the locked raw code directly and does not need embeddings.

In [1]:
from pathlib import Path
import hashlib
import json
import os
import tempfile
import numpy as np

ROOT = Path('/mnt/d/JUNIOR/LabThayTho/GRACE-summary')
STEP6 = ROOT / 'STEP6_supervised_anchors'
LOCK_PATH = STEP6 / 'step6_lock.json'
LOCK_SHA256 = '64fb2c99d9044548aaa07e949f1cd710580ff1b11571fdb833ddc18a56f80575'
assert ROOT.is_dir(), f'WSL repo path is unavailable: {ROOT}'

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(chunk_size), b''):
            h.update(block)
    return h.hexdigest()

assert sha256_file(LOCK_PATH) == LOCK_SHA256, 'Committed lock hash mismatch; stop.'
LOCK = json.loads(LOCK_PATH.read_text(encoding='utf-8'))
assert LOCK['schema'] == 'grace_step6_execution_lock_v1'
assert LOCK['embedding_cache_audit']['cache_used_for_final_prompt_retrieval']['rebuild_required'] is False
print('Loaded committed Step 6 lock:', LOCK_PATH)
print('Lock SHA-256:', LOCK_SHA256)

Loaded committed Step 6 lock: /mnt/d/JUNIOR/LabThayTho/GRACE-summary/STEP6_supervised_anchors/step6_lock.json
Lock SHA-256: 64fb2c99d9044548aaa07e949f1cd710580ff1b11571fdb833ddc18a56f80575


In [2]:
def has_ref(node, path, digest):
    if isinstance(node, dict):
        if node.get('path') == path and node.get('sha256') == digest:
            return True
        return any(has_ref(v, path, digest) for v in node.values())
    if isinstance(node, list):
        return any(has_ref(v, path, digest) for v in node)
    return False

# Verify the registered protocol, frozen source manifest, and active prompt-to-retrieval chain.
protocol = LOCK['protocol']
assert sha256_file(ROOT / protocol['protocol_file']) == protocol['protocol_sha256']
freeze = LOCK['frozen_data']['freeze_manifest']
assert sha256_file(ROOT / freeze['path']) == freeze['sha256']
binding = LOCK['embedding_cache_audit']['cache_used_for_final_prompt_retrieval']['binding_to_active_prompts']
active = binding['active_manifest']
active_path = ROOT / active['path']
assert sha256_file(active_path) == active['sha256']
active_obj = json.loads(active_path.read_text(encoding='utf-8'))
for key in ('devign_retrieval_provenance', 'reveal_retrieval_provenance'):
    ref = binding[key]
    assert ref['sha256'] == ref['active_manifest_reference_sha256']
    assert sha256_file(ROOT / ref['path']) == ref['sha256']
    assert has_ref(active_obj, ref['path'], ref['sha256']), f'Active manifest does not reference {ref["path"]}'
    print('Active manifest references verified:', ref['path'])
print('Protocol, freeze manifest, and prompt provenance chain verified.')

Active manifest references verified: Runs-management/donor_reconciliation_20261101/devign_retrieval_provenance.json
Active manifest references verified: Runs-management/donor_reconciliation_20261101/reveal_retrieval_provenance.json
Protocol, freeze manifest, and prompt provenance chain verified.


In [3]:
CACHE_ROOT = ROOT / '01_Author_Baseline_Unified' / 'output' / 'local_full'
CACHE_LOCK = LOCK['embedding_cache_audit']['cache_used_for_final_prompt_retrieval']
EXPECTED_MODEL = LOCK['protocol']['codet5']['model'] + '@' + LOCK['protocol']['codet5']['revision']

def code_sha(code):
    return hashlib.sha256(code.encode('utf-8')).hexdigest()

def frozen_id_hashes(raw_path):
    pairs = []
    seen = set()
    with Path(raw_path).open(encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            rid = str(row['id'])
            if rid in seen:
                raise ValueError(f'Duplicate frozen ID: {rid}')
            seen.add(rid)
            pairs.append({'id': rid, 'code_sha256': code_sha(str(row['func']).strip())})
    return pairs

checked = {}
for dataset, locked in LOCK['frozen_data']['datasets'].items():
    cache_ds = CACHE_LOCK['datasets'][dataset]
    provenance_ref = binding[f'{dataset}_retrieval_provenance']
    provenance = json.loads((ROOT / provenance_ref['path']).read_text(encoding='utf-8'))
    provenance_splits = {item['split']: item for item in provenance['codet5_cache']}
    donor = binding['clean_donor_banks'][dataset]
    donor_new = provenance['new']
    assert donor_new['input_sha256'] == locked['frozen_donor_bank']['sha256']
    assert donor_new['cache_sha256'] == donor['sha256']
    for split in ('train', 'validation', 'test'):
        raw = locked['raw'][split]
        split_manifest = locked['manifests'][split]
        assert sha256_file(ROOT / raw['path']) == raw['sha256'], f'{dataset}/{split}: raw hash mismatch'
        assert sha256_file(ROOT / split_manifest['path']) == split_manifest['sha256'], f'{dataset}/{split}: frozen manifest mismatch'
        spec = cache_ds['splits'][split]
        folder = CACHE_ROOT / dataset / 'embedding_cache' / split
        manifest_path, vectors_path = folder / 'manifest.json', folder / 'vectors.npy'
        assert sha256_file(manifest_path) == spec['manifest_sha256'], f'{dataset}/{split}: cache manifest hash mismatch'
        assert sha256_file(vectors_path) == spec['vectors_sha256'], f'{dataset}/{split}: vector hash mismatch'
        meta = json.loads(manifest_path.read_text(encoding='utf-8'))
        assert meta['model'] == EXPECTED_MODEL and meta['completed'] == len(meta['rows'])
        assert meta['max_length'] == LOCK['protocol']['codet5']['max_length']
        assert meta['pooling'] == LOCK['protocol']['codet5']['pooling']
        assert provenance_splits[split]['manifest_sha256'] == spec['manifest_sha256']
        assert provenance_splits[split]['vectors_sha256'] == spec['vectors_sha256']
        expected_rows = frozen_id_hashes(ROOT / raw['path'])
        assert meta['rows'] == expected_rows, f'{dataset}/{split}: ID/code identity differs from frozen source'
        vectors = np.load(vectors_path, mmap_mode='r')
        assert vectors.shape == (len(expected_rows), LOCK['protocol']['codet5']['dimension'])
        assert vectors.dtype == np.float32 and np.isfinite(vectors).all()
        checked[dataset, split] = (expected_rows, vectors, spec)
        print(f'PASS {dataset}/{split}: {len(expected_rows)} rows; exact locked cache and frozen identity')
print('All six locked CodeT5 caches passed pre-export checks; no encoder was called.')

PASS devign/train: 21854 rows; exact locked cache and frozen identity
PASS devign/validation: 2732 rows; exact locked cache and frozen identity
PASS devign/test: 2732 rows; exact locked cache and frozen identity
PASS reveal/train: 18187 rows; exact locked cache and frozen identity
PASS reveal/validation: 2273 rows; exact locked cache and frozen identity
PASS reveal/test: 2274 rows; exact locked cache and frozen identity
All six locked CodeT5 caches passed pre-export checks; no encoder was called.


In [4]:
# Export the existing mmap vectors and the verified IDs as NPZ for supervised_anchor.py.
OUT = STEP6 / 'cache' / 'anchor_npz'
OUT.mkdir(parents=True, exist_ok=True)
exports = []
for (dataset, split), (rows, vectors, source_spec) in checked.items():
    destination = OUT / dataset / f'{split}.npz'
    destination.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary = tempfile.mkstemp(prefix=destination.name + '.', suffix='.tmp', dir=destination.parent)
    try:
        with os.fdopen(fd, 'wb') as f:
            np.savez(f, ids=np.asarray([row['id'] for row in rows], dtype=np.str_), X=vectors)
            f.flush()
            os.fsync(f.fileno())
        os.replace(temporary, destination)
    except Exception:
        if os.path.exists(temporary):
            os.unlink(temporary)
        raise
    with np.load(destination, allow_pickle=False) as exported:
        assert exported['ids'].tolist() == [row['id'] for row in rows]
        assert exported['X'].shape == vectors.shape and exported['X'].dtype == np.float32
    exports.append({
        'dataset': dataset, 'split': split, 'path': str(destination.relative_to(ROOT)),
        'rows': len(rows), 'shape': list(vectors.shape),
        'source_manifest_sha256': source_spec['manifest_sha256'],
        'source_vectors_sha256': source_spec['vectors_sha256'],
        'npz_sha256': sha256_file(destination)
    })
    print('Wrote ID-aligned adapter:', destination, len(rows), vectors.shape)

adapter_manifest = {
    'schema': 'step6_locked_codet5_npz_adapter_v1',
    'lock_path': str(LOCK_PATH.relative_to(ROOT)),
    'lock_sha256': LOCK_SHA256,
    'lock_commit': 'e8f3cdc',
    'encoder_rerun': False,
    'model': EXPECTED_MODEL,
    'dimension': LOCK['protocol']['codet5']['dimension'],
    'exports': exports
}
manifest_out = OUT / 'adapter_manifest.json'
manifest_out.write_text(json.dumps(adapter_manifest, indent=2) + '\n', encoding='utf-8')
print('Adapter manifest:', manifest_out)

Wrote ID-aligned adapter: /mnt/d/JUNIOR/LabThayTho/GRACE-summary/STEP6_supervised_anchors/cache/anchor_npz/devign/train.npz 21854 (21854, 768)
Wrote ID-aligned adapter: /mnt/d/JUNIOR/LabThayTho/GRACE-summary/STEP6_supervised_anchors/cache/anchor_npz/devign/validation.npz 2732 (2732, 768)
Wrote ID-aligned adapter: /mnt/d/JUNIOR/LabThayTho/GRACE-summary/STEP6_supervised_anchors/cache/anchor_npz/devign/test.npz 2732 (2732, 768)
Wrote ID-aligned adapter: /mnt/d/JUNIOR/LabThayTho/GRACE-summary/STEP6_supervised_anchors/cache/anchor_npz/reveal/train.npz 18187 (18187, 768)
Wrote ID-aligned adapter: /mnt/d/JUNIOR/LabThayTho/GRACE-summary/STEP6_supervised_anchors/cache/anchor_npz/reveal/validation.npz 2273 (2273, 768)
Wrote ID-aligned adapter: /mnt/d/JUNIOR/LabThayTho/GRACE-summary/STEP6_supervised_anchors/cache/anchor_npz/reveal/test.npz 2274 (2274, 768)
Adapter manifest: /mnt/d/JUNIOR/LabThayTho/GRACE-summary/STEP6_supervised_anchors/cache/anchor_npz/adapter_manifest.json


## Output and next step

The six adapters are written under `STEP6_supervised_anchors/cache/anchor_npz/{devign,reveal}/{train,validation,test}.npz`, each with `ids` and `X`; `adapter_manifest.json` records source and output hashes. The source `vectors.npy`, frozen data, donor banks, prompt bundles, and lock are read-only in this notebook.

These NPZ files are ready for the later anchor-fitting step. Fitting must still follow the lock: seeds 42/43/44, fit on train, choose thresholds on validation by MCC, and do not use test labels for fitting or selection.